# 03 — Feature Engineering

Builds the five engineered features via `BNPLFeatureBuilder` and shows how `risk_scope` changes which columns actually reach the model.

In [1]:
from bnpl_credit_risk.data.loaders import BNPLDataLoader
from bnpl_credit_risk.features.builder import BNPLFeatureBuilder, resolve_feature_columns
from bnpl_credit_risk.settings import get_settings, load_config

settings = get_settings()
config = load_config()
df = BNPLDataLoader(settings, config.data).load_raw()
engineered = BNPLFeatureBuilder(config.features).transform(df)
engineered[['payment_stress', 'income_to_purchase_ratio', 'age_group', 'txn_month', 'is_high_risk']].head()

2026-07-13 08:36:13.393 | INFO     | bnpl_credit_risk.data.loaders:load_raw:37 - Loaded raw dataset path=/Users/surelmanda/BNPL-Credit-Risk/data/raw/BNPL_CreditRisk_Dataset.csv rows=10345 columns=17


,payment_stress,income_to_purchase_ratio,age_group,txn_month,is_high_risk
0,13,13.703159,46-59,6,0
1,13,6.747019,18-25,10,1
2,38,8.314789,18-25,4,0
3,90,3.537043,18-25,6,1
4,0,8.567387,36-45,10,0


## application_risk vs behavioral_risk

`resolve_feature_columns` is what `configs/model.yaml`'s `risk_scope` drives at train time — this is the mechanism that keeps leaky features out of the production (application_risk) model.

In [2]:
app_numeric, app_categorical = resolve_feature_columns(config.features, 'application_risk')
beh_numeric, beh_categorical = resolve_feature_columns(config.features, 'behavioral_risk')
print('application_risk numeric:', app_numeric)
print('behavioral_risk numeric adds:', sorted(set(beh_numeric) - set(app_numeric)))

application_risk numeric: ['age', 'monthly_income', 'credit_score', 'purchase_amount', 'bnpl_installments', 'app_usage_frequency', 'debt_to_income_ratio', 'income_to_purchase_ratio', 'txn_month']
behavioral_risk numeric adds: ['is_high_risk', 'missed_payments', 'payment_stress', 'repayment_delay_days', 'risk_score']
